In [1]:
from aig_grapher import AIG
from typing import Dict, Set, List, Tuple, Any
from colorama import Fore

In [2]:
from pydantic_ai import Agent
from pydantic_ai import RunContext, UsageLimits
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

local_is_cloud = True
local_model_name = 'nemotron-3-super:cloud' if local_is_cloud else 'glm-4.7-flash'
cloud_model_name = "minimax-m2.7:cloud"	

local_model = OpenAIChatModel(
	model_name=local_model_name,
	provider=OllamaProvider(base_url='http://localhost:11434/v1' if local_is_cloud else 'http://kfed:11434/v1'),  
)

cloud_model = OpenAIChatModel(
	model_name=cloud_model_name,
	provider=OllamaProvider(base_url='http://localhost:11434/v1'),  
)

# local_model = cloud_model

In [3]:
from ollama import Client
import os

os.environ["OLLAMA_API_KEY"] = "24691ae7a5aa4008b278b5859c17e1ec.kxf5OcJXXzJ9Ies9xlkjJlRr"
client = Client(
	host="https://ollama.com",
	# host="http://kfed:11434",
	headers={'Authorization': 'Bearer ' + str(os.environ.get('OLLAMA_API_KEY'))}
)

def query_llm(prompt: str):
	messages = [
	{
		'role': 'user',
		'content': prompt,
	},
	]

	res = client.chat("minimax-m2.7:cloud", messages=messages, stream=False)
	return res


In [ ]:
verilog_path = '/home/krishnendu/Research/fv-invariant-mining/data/circuits/verilog/multipliers.v'
print("Reading Verilog file and generating condensed technical representation using LLM...")
with open(verilog_path, 'r') as f:
	verilog_content = f.read()
	
verilog_embed = query_llm(f"You are a formal verification engineer. Analyze the two modules in this verilog code and compile an extremely condensed technical representation for use in downstream LLM-based tasks. No human is going to read that representation. So, make it as much information dense as you want. solving the miter of the two circuits. A module might be dependent on high-level syntheis techniques. Verilog code:\n{verilog_content}")
verilog_embed = str(verilog_embed.message.content)
print("Condensed technical representation generated.")

Reading Verilog file and generating condensed technical representation using LLM...
Condensed technical representation generated.


In [ ]:
verilog_embed

In [5]:
aig_agent_instr = """
You are a the participant of an And-Inverter Graph Exploration game. You are given an AIG graph of a Miter circuit in ASCII format, and you have to find meaningful equivalences between nodes of the graph. 
For every invariant or equivalence you find, you will submit it as a response to a validator, who will award you points based on the following criteria:
- If the invariant/equivalence is correct and non-trivial, you will receive 10 points.
- If the invariant/equivalence is correct but trivial, you will receive 5 points
- If the invariant/equivalence is incorrect or if proving the correctness timeouts the validator, you will receive 0 points.
DO NOT BRUTE FORCE by submitting random tuples of nodes as equivalences, as that will likely lead to timeouts and zero points. Instead, use the structure of the graph and any insights you can gather to find promising candidates for equivalences.
You are expected to find as many correct and non-trivial invariants/equivalences as possible to maximize your score. You can also find trivial equivalences, but they will yield fewer points.
IMPORTANT: 
	Since you *are* interrupted and restarted from blank state on some different graph at many points due to usage-based timeouts, it is advised to store your developed strategies and insights in persistent storage, and load them back when you are restarted, so that you can continue improving your strategies over time. 
	Always check for strategies and knowledge already stored.
	Seek help from the expert agent to score more points. You are encouaged to do so.
CONSTRAINT: You will only be allowed to generate limigted number of tokens, so do efficient work and save all learnings for reuse in later turns.
"""

expert_agent_instr = """
You are an And-Inverter Graph circuits expert whose task is to aid a novice LLM whenever it needs help. As all communications with you will be a LLM<->LLM communication, you are free to use more efficient ways of information communication.
"""

In [6]:
class AIG_data:
	def __init__(self, aig: AIG, pop_default=True):
		self.aig = aig
		self.num_nodes = len(aig.nodes)
		self.data: List[Dict[str, str]] = [{} for _ in range(self.num_nodes)]

		if pop_default:
			self.populate_default()

	def populate_data(self, indices: List[int], key: str, values: List[str]):
		assert len(indices) == len(values), "Length of indices and values must be the same."
		
		for idx, val in zip(indices, values):
			self.data[idx][key] = val

	def populate_default(self):
		self.populate_data(self.aig.input_ids, "prestored", [f"input_bit_{i}" for i in range(len(self.aig.input_ids))])
		self.populate_data([0], "prestored", ["constant_0/FALSE"])
		self.populate_data([id for id, _ in self.aig.outputs], "prestored", [f"{'INVERTED_' if inv else ''}output_bit_{i}" for i, (_, inv) in enumerate(self.aig.outputs)])

In [7]:

class Config:
	def __init__(self, aig_path: str):
		self.aig_path: str = aig_path
		self.aig: AIG = AIG(aig_path)
		self.data = AIG_data(self.aig)
		self.expert_comm_hist: List[str] = []
		self.score: int = 0
		self.opponent_stats: Tuple[int, str] = (0, "") # score, strategies
		self.strategies: List[str] = []
		self.knowledge_base: List[str] = []
		self.results: Set[Tuple[int, int]] = set()
		self.tried_tuples: Set[Tuple[int, int]] = set() 

	def dump(self, filename: str):
		"""Persist the current game state to disk.

		The dump excludes the in-memory AIG object (which is not JSON-serializable)
		and instead keeps the path to the original AIG file.
		"""
		import json

		data: Dict[str, Any] = {
			"aig_path": self.aig_path,
			"score": self.score,
			"opponent_stats": self.opponent_stats,
			"strategies": self.strategies,
			"results": [list(t) for t in self.results],
			"tried_tuples": [list(t) for t in self.tried_tuples],
			"expert_comm_hist": self.expert_comm_hist,
		}

		with open(filename, "w", encoding="utf-8") as f:
			json.dump(data, f, indent=2)

	@classmethod
	def load(cls, filename: str) -> "Config":
		"""Load a previously dumped game state from disk."""
		import json

		with open(filename, "r", encoding="utf-8") as f:
			data = json.load(f)

		conf = cls(data["aig_path"])
		conf.score = data.get("score", 0)
		conf.opponent_stats = tuple(data.get("opponent_stats", (0, "")))
		conf.strategies = data.get("strategies", [])
		conf.results = {tuple(t) for t in data.get("results", [])}
		conf.tried_tuples = {tuple(t) for t in data.get("tried_tuples", [])}
		conf.expert_comm_hist = data.get("expert_comm_hist", [])
		return conf

In [8]:
aig_agent = Agent(
	model=local_model,
	system_prompt=aig_agent_instr,
	deps_type=Config
)

expert_limits = UsageLimits(
	# request_limit=10,          # Max 10 total turns (Thinking + Tool Call + Response)
	output_tokens_limit=3000,   # Stop if the total output across all turns exceeds this value
)

expert_agent = Agent(
	model=cloud_model,
	instructions=expert_agent_instr,
	deps_type=List[str]
)

# @aig_agent.tool_plain
# def get_weather(city: str) -> int:
#     """
#     Get weather for a given city, but with a twist. The returned value is 8 * current temp (in C)
#     Args:
#         city: name of the city
#     Returns:
#         temperature
#     """
#     return 100

@aig_agent.tool
def get_verilog_context(ctx: RunContext[Config]):
	"""
	 Returns condensed information from the verilog file from which the modules have been taken of which the miter has been formed.
	"""
	return verilog_embed

# @aig_agent.tool
# def write_node_data(ctx: RunContext[Config], node: int, key: str, data: str):
# 	"""
# 	Store data for an AIG node, that you think will be useful for you during further exploration
# 	Args:
# 		node: node in AIG graph (follows same conventions as node numbers in .aag/.aig files). 
# 		key: key for the data
# 		data: associated data to store in that node for that key
# 	"""
# 	ctx.deps.data.data[node//2][key] = data

# @aig_agent.tool
# def read_node_data(ctx: RunContext[Config], node: int):
# 	"""
# 	Read data stored for an AIG node, that you stored during previous exploration
# 	Args:
# 		node: node in AIG graph (follows same conventions as node numbers in .aag/.aig files). 
# 		key: key for the data
# 		data: associated data to store in that node for that key
# 	"""
# 	return str(ctx.deps.data.data[node//2])

@aig_agent.tool
def check_equivalence_and_add(ctx: RunContext[Config], a: int, b: int) -> Tuple[int, Tuple[int, int], str]:
	"""
	Checks if nodes a and b are indeed equivalent and returns a score based on the correctness and triviality of the finding.
	Also, if indeed equivalent, this function stores the equivalence and updates the score.
	Every failed use, incurs a -3 penalty.
	Args:
		a, b: nodes in AIG graph (follows same conventions as node numbers in .aag/.aig files). 
	Returns:
		score: the reward/penalty for the equivalence check submission.
		simulation_tuple: (sim_val_a, sim_val_b) where each value is a N-bit integer corresponding to the value calculated for nodes a and b during a parallel N-bit simulation on the AIG. Returns negative values if function run on illegal inputs.
		err_msg: error message if any
	"""
	conf = ctx.deps
	eqv_tuple = (a, b) if a > b else (b, a)
	err_msg_success = "please store your strategy asap"
	try:
		assert a != b, "same input node ids"
		assert eqv_tuple not in conf.tried_tuples, "simulation already tried for pair"

		conf.tried_tuples.add(eqv_tuple)
		aig = conf.aig 
		nv, _ = aig.simulate(128)
		sim_val_a = nv[a//2]
		sim_val_b = nv[b//2]
		if a % 2 == 1:
			sim_val_a = sim_val_a ^ ((1 << 32) - 1)
		if b % 2 == 1:
			sim_val_b = sim_val_b ^ ((1 << 32) - 1)
		score_change = 10 if sim_val_a == sim_val_b else 0
		conf.score += score_change
		if score_change != 0:
			conf.results.add(eqv_tuple)
		return score_change, (sim_val_a, sim_val_b), err_msg_success if score_change != 0 else "failed"
	except Exception as e:
		conf.score -= 3
		return 0, (-1, -1), str(type(e)) + " " + str(e) + " | If you find any reason for your failure, add that to knowledge_base"
  
@aig_agent.tool
def expert_comm_history(ctx: RunContext[Config]) -> List[str]:
	"""
	Get your history of communication with the expert.
	"""
	return ctx.deps.expert_comm_hist


@aig_agent.tool
async def expert_help(ctx: RunContext[Config], prompt: str, data: str) -> str:
	"""
	Allows you to send a prompt to an expert for support when you are stuck or confused.
	Args:
		prompt: your question/doubts/queries. Make sure to tell the expert that you need the data for use by another LLM and not a human so that the communication is more efficient.
		data: any data the expert needs. Don't assume the LLM knows what you are solving.
	"""
	final_prompt = f"{prompt}\nDATA:\n{data}" 
	print(f"{Fore.GREEN} Using expert help using prompt: {final_prompt}")
	ctx.deps.expert_comm_hist.append("USER: " + prompt)
	result = await expert_agent.run(final_prompt,usage_limits=expert_limits, deps=ctx.deps.expert_comm_hist)
	ctx.deps.expert_comm_hist.append("EXPERT: " + result.output)
	return result.output

@expert_agent.tool
async def get_comm_history(ctx: RunContext[List[str]]) -> List[str]:
	"""
	Get your history in interacting with the user.
	"""
	return ctx.deps

@aig_agent.tool
def score(ctx: RunContext[Config]) -> int:
	"""
	Returns your current game score that you have to maximize
	"""
	return ctx.deps.score


@aig_agent.tool
def results(ctx: RunContext[Config]):
	"""
	Returns your currently found equivalences/invariants.
	"""
	return ctx.deps.results

@aig_agent.tool
def append_strategies(ctx: RunContext[Config], strategy: List[str]):
	"""
	Store new strategies in your private knowledge that helped you to find equivalences, or things that you learnt along the way by failures. This information is AIG instance-agnostic.
	This information must not refer to instance-specific data, such data goes to your knowledge base.
	"""
	ctx.deps.strategies.extend(strategy)

@aig_agent.tool
def strategies(ctx: RunContext[Config]) -> List[str]:
	"""
	Returns the stored AIG instance-agnostic strategies from this and past runs.
	"""
	return ctx.deps.strategies

@aig_agent.tool
def knowledge_base(ctx: RunContext[Config]) -> List[str]:
	"""
	Returns your instance-specific stored knowledge base from this and past runs
	"""
	return ctx.deps.knowledge_base

@aig_agent.tool
def append_knowledge_base(ctx: RunContext[Config], knowledge: List[str]):
	"""
	Store new instance-specific knowledge in your private knowledge base that applies on this AIG graph
	Usage direction: Whenever you find any information, trivial or not, but important enough to get up to work faster next time you are started, store it here.
	"""
	ctx.deps.knowledge_base.extend(knowledge)

In [9]:
def read_file_as_string(filepath: str) -> str:
	with open(filepath, "r", encoding="utf-8") as f:
		return f.read()

In [10]:
from pathlib import Path


aig_path = "/home/krishnendu/Research/fv-invariant-mining/data/circuits/aag/miter_addr_5bit.aag"
aig_str = read_file_as_string(aig_path)
conf = Config(aig_path)

stored_conf = Config(aig_path)
stored_conf_path = Path('/home/krishnendu/Research/fv-invariant-mining/notebooks/conf.dump')
if stored_conf_path.is_file():
    stored_conf.load(str(stored_conf_path))
    conf.strategies = stored_conf.strategies

In [11]:
events: List[Any] = []
# conf = old_conf
# conf.strategies.extend(old_conf.strategies)

async for event in aig_agent.run_stream_events(aig_str, deps=conf, usage_limits=UsageLimits(request_limit=100)):
	events.append(event)
	if event.event_kind != 'part_delta':
		print(event)


CancelledError: 

In [49]:
print("Score\t\t:", conf.score)
print("Results\t\t:", conf.results)
print("Tried\t\t:", conf.tried_tuples)
print("Strategies\t:", conf.strategies)
print("Knowledge\t:", conf.knowledge_base)

Score		: -9
Results		: set()
Tried		: {(28, 26), (58, 22), (4, 2), (34, 32), (24, 22), (1, 0)}
Strategies	: []
Knowledge	: []


In [32]:
conf.dump('conf.dump')

In [31]:
len(conf.results) / len(conf.tried_tuples)

0.2631578947368421

In [13]:
old_conf = conf

In [ ]:
str({"a": 4, "fds": "gfds"})